In [ ]:
!ls /Users/karella/Projects/rotation-invariant-neural-networks/hippy2d/logs/inv_factor

In [ ]:
from glob import glob
import yaml
import pandas as pd
dir_root = "/Users/karella/Projects/rotation-invariant-neural-networks/hippy2d/logs/inv_factor"
dir = "prototype-mnistrottest-copy-preserveenergy"

results = []

for dir in glob(f"{dir_root}/*"):
    for run in glob(f"{dir}/*"):
        print(run)
        
        # Load config first
        with open(f"{run}/hparams.yaml", 'r') as file:
            config = yaml.safe_load(file)
        
        metrics = pd.read_csv(f"{run}/metrics.csv")
        rci_norm = metrics['test_rci_norm'].dropna().values[0]
        test_acc = metrics['test_acc'].dropna().values[0]
        rci_sim = metrics['test_rci_sim'].dropna().values[0]
        train_acc = metrics['train_acc'].max()
        valid_45_acc = metrics['val_45_acc'].max()
        valid_90_acc = metrics['val_90_acc'].max()
        valid_acc = metrics['val_acc'].max()
        invariant_factor = config['m_param']['layer_kwargs']['magnitude_normalization']
        preserve_energy = config['m_param']['layer_kwargs'].get('preserve_energy', False)
        zero_out_middles = config['m_param']['layer_kwargs'].get('zero_out_middles', False)
        middle_only = run.split('/')[-2].endswith('-just-middle')
        print(f"rci_norm: {rci_norm}, rci_sim: {rci_sim}, test_acc: {test_acc}, train_acc: {train_acc}, valid_45_acc: {valid_45_acc}, valid_90_acc: {valid_90_acc}, valid_acc: {valid_acc}, invariant_factor: {invariant_factor}, preserve_energy: {preserve_energy}")
        
        # Save results to dictionary
        results.append({
            'run': run,
            'rci_norm': rci_norm,
            'rci_sim': rci_sim,
            'test_acc': test_acc,
            'train_acc': train_acc,
            'valid_45_acc': valid_45_acc,
            'valid_90_acc': valid_90_acc,
            'valid_acc': valid_acc,
            'zero_out_middles': zero_out_middles,
            'middle_only': middle_only,
            'invariant_factor': invariant_factor,
            'preserve_energy': preserve_energy
        })

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
df = pd.DataFrame(results)
fig, axs = plt.subplots(1,1)
sns.swarmplot(data=df.loc[(df['zero_out_middles'] == False) & (df['middle_only'] == False) & (df['preserve_energy'] == False)],
                 x='invariant_factor',
                y='test_acc',
                s=10) 

# rename x labels
axs.set_xticklabels(['r^(p-q)', '1', 'r'])


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
df = pd.DataFrame(results)
fig, axs = plt.subplots(1,1)
sns.swarmplot(data=df.loc[(df['zero_out_middles'] == True) & (df['middle_only'] == True)],
                 x='invariant_factor',
                y='test_acc',
                hue='preserve_energy',
                s=10,
                ax=axs) 

sns.swarmplot(data=df.loc[(df['zero_out_middles'] == False) & (df['middle_only'] == False)],
                 x='invariant_factor',
                y='test_acc',
                hue='preserve_energy',
                color='gray',
                s=10,
                ax=axs) 

# rename x labels
axs.set_xticklabels(['r^(p-q)', '1', 'r'])


In [ ]:
import torch 
jump = lambda x: torch.nn.functional.tanh(torch.tensor(x))/2.0 + 0.5

In [ ]:
# plot jump function
x = torch.linspace(-100, 100, 10000)
y = jump(x)
plt.plot(x, y)

In [ ]:
jump(-10), jump(0), jump(0.5), jump(1)

In [ ]:
import torch

import torch

def smoothstep(x, edge0=1e-3, edge1=1e-1):
    # edge0: where output starts rising from 0
    # edge1: where output reaches 1
    t = ((x - edge0) / (edge1 - edge0)).clamp(0.0, 1.0)
    return t * t * (3 - 2*t)




In [ ]:
# Plot smoothstep function
x = torch.linspace(-0.1, 0.1, 10000)
y = smoothstep(x)
plt.plot(x, y)